# Lab 1 — Introduction to LLMs: Practical Activities

**Student:** Atherv Telkar  
**Roll No:** SCFP125043  
**Repository:** `GenAI_LLM_engineering_SCFP125043`

This notebook covers all 10 activities from Lab 1, using Hugging Face Transformers with a local GPU (NVIDIA RTX 4050, 6 GB VRAM).

## Setup — Install & Import Libraries

In [1]:
import torch
import psutil
import platform
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from huggingface_hub import HfApi

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1)
    print(f"VRAM: {vram_gb} GB")

ram_gb = round(psutil.virtual_memory().total / 1024**3, 1)
print(f"System RAM: {ram_gb} GB")
print(f"OS: {platform.system()} {platform.release()}")

d:\Anaconda\envs\advanced_ml\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
VRAM: 6.0 GB
System RAM: 15.7 GB
OS: Windows 10


---
## Activity 1 — Explore LLM Models on Hugging Face

**Task:** Pick three models on Hugging Face — one small, one medium, one large — and answer the guided questions.

In [2]:
api = HfApi()

models_to_explore = [
    "distilgpt2",              # Small
    "gpt2-medium",             # Medium
    "meta-llama/Llama-3.2-1B", # Large
]

for model_id in models_to_explore:
    try:
        info = api.model_info(model_id)
        print(f"{'='*60}")
        print(f"Model: {info.modelId}")
        print(f"Author/Creator: {info.author or 'Unknown'}")
        
        # Parameter count
        if hasattr(info, 'safetensors') and info.safetensors:
            params = info.safetensors.get('total', 'N/A')
            if isinstance(params, int):
                print(f"Parameters: {params:,}")
            else:
                print(f"Parameters: {params}")
        else:
            print("Parameters: Check model card")
        
        # Tags
        tags = info.tags or []
        print(f"Tags: {tags[:10]}")
        
        # Is it generative?
        is_generative = any(
            t in tags
            for t in ["text-generation", "causal-lm"]
        )
        print(f"Is Generative (text-generation): {is_generative}")
        
        # Pipeline tag
        print(f"Pipeline Tag: {info.pipeline_tag}")
        
        # Downloads
        print(f"Downloads (last month): {info.downloads:,}")
        print()
    except Exception as e:
        print(f"Could not fetch info for {model_id}: {e}")
        print()

Model: distilbert/distilgpt2
Author/Creator: distilbert
Parameters: 88,204,032
Tags: ['transformers', 'pytorch', 'tf', 'jax', 'tflite', 'rust', 'coreml', 'safetensors', 'gpt2', 'text-generation']
Is Generative (text-generation): True
Pipeline Tag: text-generation
Downloads (last month): 1,874,818

Model: openai-community/gpt2-medium
Author/Creator: openai-community
Parameters: 379,988,992
Tags: ['transformers', 'pytorch', 'tf', 'jax', 'rust', 'onnx', 'safetensors', 'gpt2', 'text-generation', 'en']
Is Generative (text-generation): True
Pipeline Tag: text-generation
Downloads (last month): 218,633

Model: meta-llama/Llama-3.2-1B
Author/Creator: meta-llama
Parameters: 1,235,814,400
Tags: ['transformers', 'safetensors', 'llama', 'text-generation', 'facebook', 'meta', 'pytorch', 'llama-3', 'en', 'de']
Is Generative (text-generation): True
Pipeline Tag: text-generation
Downloads (last month): 682,413



### Activity 1 — Answers

| Question | distilgpt2 (Small) | gpt2-medium (Medium) | Llama-3.2-1B (Large) |
|---|---|---|---|
| **Creator** | Hugging Face | OpenAI | Meta (Facebook AI) |
| **Parameters** | ~82M | ~355M | ~1.24B |
| **Generative?** | Yes (causal LM) | Yes (causal LM) | Yes (causal LM) |
| **Context Length** | 1024 tokens | 1024 tokens | 131,072 tokens |
| **Suitable for local execution?** | Yes — very lightweight | Yes — fits in 4 GB VRAM | Possible with quantization on 6 GB VRAM |
| **Recommended Hardware** | CPU or any GPU | GPU with ≥4 GB VRAM | GPU with ≥6 GB VRAM (FP16) or quantized |

**Key Observations:**
- Smaller models like distilgpt2 can run on almost any hardware, even a CPU.
- Medium models like GPT-2 Medium are still manageable on consumer GPUs.
- Large models like Llama-3.2-1B require GPU acceleration and may need quantization (INT4/INT8) to fit in limited VRAM.

---
## Activity 2 — Tokenization Practical

**Task:** Tokenize a sentence, then try different words and phrases. Answer the guided questions.

In [3]:
import tiktoken

# Using tiktoken (GPT-4 tokenizer)
enc = tiktoken.get_encoding("cl100k_base")

sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "Artificial Intelligence",
    "AI",
    "unbelievable",
    "I can't believe it!",
    "Hello, world!",
    "supercalifragilisticexpialidocious",
    "123456789",
]

print("=" * 70)
print(f"{'Text':<45} {'Tokens':<8} Token IDs")
print("=" * 70)

for sentence in sentences:
    tokens = enc.encode(sentence)
    decoded_tokens = [enc.decode([t]) for t in tokens]
    print(f"{sentence:<45} {len(tokens):<8} {decoded_tokens}")

print()

Text                                          Tokens   Token IDs
The quick brown fox jumps over the lazy dog.  10       ['The', ' quick', ' brown', ' fox', ' jumps', ' over', ' the', ' lazy', ' dog', '.']
Artificial Intelligence                       3        ['Art', 'ificial', ' Intelligence']
AI                                            1        ['AI']
unbelievable                                  3        ['un', 'belie', 'vable']
I can't believe it!                           6        ['I', ' can', "'t", ' believe', ' it', '!']
Hello, world!                                 4        ['Hello', ',', ' world', '!']
supercalifragilisticexpialidocious            11       ['sup', 'erc', 'al', 'if', 'rag', 'il', 'istic', 'exp', 'ial', 'id', 'ocious']
123456789                                     3        ['123', '456', '789']



In [4]:
# Let's also see the HuggingFace GPT-2 tokenizer for comparison
from transformers import AutoTokenizer

tokenizer_gpt2 = AutoTokenizer.from_pretrained("gpt2")

test_words = ["running", "unbelievable", "ChatGPT", "don't", "Hello!", "  spaces  "]

print("GPT-2 Tokenizer Breakdown:")
print("=" * 60)

for word in test_words:
    tokens = tokenizer_gpt2.tokenize(word)
    ids = tokenizer_gpt2.encode(word)
    print(f"Word: '{word}'")
    print(f"  Sub-word tokens: {tokens}")
    print(f"  Token IDs: {ids}")
    print()

GPT-2 Tokenizer Breakdown:
Word: 'running'
  Sub-word tokens: ['running']
  Token IDs: [20270]

Word: 'unbelievable'
  Sub-word tokens: ['un', 'bel', 'iev', 'able']
  Token IDs: [403, 6667, 11203, 540]

Word: 'ChatGPT'
  Sub-word tokens: ['Chat', 'G', 'PT']
  Token IDs: [30820, 38, 11571]

Word: 'don't'
  Sub-word tokens: ['don', "'t"]
  Token IDs: [9099, 470]

Word: 'Hello!'
  Sub-word tokens: ['Hello', '!']
  Token IDs: [15496, 0]

Word: '  spaces  '
  Sub-word tokens: ['Ġ', 'Ġspaces', 'Ġ', 'Ġ']
  Token IDs: [220, 9029, 220, 220]



### Activity 2 — Answers

**1. Is one word always one token?**  
No. Common short words like "the", "is", "AI" are single tokens, but longer or uncommon words get split into multiple sub-word tokens (e.g., "unbelievable" → ["un", "believ", "able"]).

**2. Why are some words split?**  
Tokenizers use sub-word algorithms (like BPE — Byte Pair Encoding). They learn the most frequent character sequences from training data. Rare or long words are broken into smaller, more common pieces. This allows the model to handle any word — even ones it has never seen before — by combining known sub-words.

**3. What happens to punctuation?**  
Punctuation marks are usually their own tokens (e.g., "!" is a separate token, "," is a separate token). Sometimes punctuation attached to a word stays together (like "'t" in "don't"), depending on the tokenizer.

**4. Why does the model need token IDs?**  
Neural networks only work with numbers, not text. Token IDs are integer indices into a vocabulary table. Each ID maps to an embedding vector that the model uses for computation. The model processes these numerical representations, not raw text.

---
## Activity 3 — Run an LLM Using GPU

**Task:** Enable GPU and run a small model. Answer the guided questions.

In [5]:
# Load distilgpt2 — a small, fast model
model_name = "distilgpt2"

print(f"Loading model: {model_name}")
print(f"Device: {device}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

print(f"\nModel loaded on: {next(model.parameters()).device}")

# Check GPU memory usage
if torch.cuda.is_available():
    mem_allocated = torch.cuda.memory_allocated() / 1024**2
    mem_reserved = torch.cuda.memory_reserved() / 1024**2
    print(f"GPU Memory Allocated: {mem_allocated:.1f} MB")
    print(f"GPU Memory Reserved: {mem_reserved:.1f} MB")

Loading model: distilgpt2
Device: cuda

Model loaded on: cuda:0
GPU Memory Allocated: 319.2 MB
GPU Memory Reserved: 356.0 MB


In [6]:
# Generate text
prompt = "Artificial intelligence will change the world by"

inputs = tokenizer(prompt, return_tensors="pt").to(device)

print(f"Prompt: {prompt}")
print(f"Input token count: {inputs['input_ids'].shape[1]}")

# Generate with different output lengths
for max_new_tokens in [20, 50, 100]:
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"\n--- max_new_tokens={max_new_tokens} ---")
    print(generated_text)

# Memory after generation
if torch.cuda.is_available():
    mem_after = torch.cuda.memory_allocated() / 1024**2
    print(f"\nGPU Memory After Generation: {mem_after:.1f} MB")

Prompt: Artificial intelligence will change the world by
Input token count: 8


d:\Anaconda\envs\advanced_ml\lib\site-packages\transformers\integrations\sdpa_attention.py:96: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(



--- max_new_tokens=20 ---
Artificial intelligence will change the world by 2025, according to a new report by the US-based think tank.






--- max_new_tokens=50 ---
Artificial intelligence will change the world by 2030. The goal is to improve our lives, and to make our world better. The goal is to help us achieve our goals.
























--- max_new_tokens=100 ---
Artificial intelligence will change the world by 2050.



































































































GPU Memory After Generation: 328.0 MB


### Activity 3 — Answers

**1. Is the GPU actually being used?**  
Yes. The model was explicitly moved to CUDA (`model.to("cuda")`), and the input tensors are also on the GPU. The `torch.cuda.memory_allocated()` shows memory consumption on the GPU, confirming it is actively being used.

**2. How much memory does the model consume?**  
distilgpt2 (~82M parameters) consumes approximately 330 MB of GPU memory in FP32. This is a very small footprint compared to the 6 GB VRAM available on the RTX 4050.

**3. What happens if you increase the output length?**  
Increasing `max_new_tokens` produces longer outputs. The model generates text one token at a time in a loop, so longer outputs take proportionally more time. Memory usage also increases slightly because the model needs to store the growing sequence of key-value caches (KV cache) for attention.

**4. Would this run on a laptop without a GPU?**  
Yes, distilgpt2 is small enough (~330 MB) to run on CPU with just system RAM. It would be slower (seconds instead of milliseconds per token), but perfectly functional. Larger models (7B+) would be impractical on CPU.

---
## Activity 4 — Can My Computer Run an LLM?

**Task:** Check your RAM and GPU, then estimate memory needs at different precisions.

In [7]:
import psutil
import torch

# System specs
ram_bytes = psutil.virtual_memory().total
ram_gb = ram_bytes / (1024**3)
print(f"System RAM: {ram_gb:.1f} GB")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_bytes = torch.cuda.get_device_properties(0).total_memory
    vram_gb = vram_bytes / (1024**3)
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram_gb:.1f} GB")

# Memory estimation for different model sizes and precisions
print("\n" + "=" * 70)
print(f"{'Model Size':<15} {'FP32':<12} {'FP16':<12} {'INT8':<12} {'INT4':<12}")
print("=" * 70)

for params_b, label in [(1, "1B"), (4, "4B"), (7, "7B"), (13, "13B"), (70, "70B")]:
    fp32 = params_b * 4   # 4 bytes per param
    fp16 = params_b * 2   # 2 bytes per param
    int8 = params_b * 1   # 1 byte per param
    int4 = params_b * 0.5 # 0.5 bytes per param
    print(f"{label:<15} {fp32:.1f} GB{'':<5} {fp16:.1f} GB{'':<5} {int8:.1f} GB{'':<5} {int4:.1f} GB")

print()
print("Note: These are estimates for model weights only.")
print("Actual usage includes KV cache, activations, and framework overhead (~20-30% extra).")

System RAM: 15.7 GB
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
VRAM: 6.0 GB

Model Size      FP32         FP16         INT8         INT4        
1B              4.0 GB      2.0 GB      1.0 GB      0.5 GB
4B              16.0 GB      8.0 GB      4.0 GB      2.0 GB
7B              28.0 GB      14.0 GB      7.0 GB      3.5 GB
13B             52.0 GB      26.0 GB      13.0 GB      6.5 GB
70B             280.0 GB      140.0 GB      70.0 GB      35.0 GB

Note: These are estimates for model weights only.
Actual usage includes KV cache, activations, and framework overhead (~20-30% extra).


### Activity 4 — Answers

**1. Your laptop has 8 GB RAM — 4B FP16, 4B INT4, or 7B INT4? Why?**  
- **4B FP16** = ~8 GB (weights only) — would NOT fit because the OS and other processes need ~2-3 GB, leaving only ~5-6 GB free.
- **4B INT4** = ~2 GB — YES, this fits comfortably with room to spare for KV cache and overhead.
- **7B INT4** = ~3.5 GB — YES, this fits but with less headroom. It should work but might be slow with limited remaining memory for KV cache.

**Best choice:** 4B INT4 for reliability, or 7B INT4 if you want a more capable model and can accept slightly tighter memory constraints.

**2. What else besides model weights uses memory?**  
- **KV Cache:** Stores attention key/value pairs for all generated tokens. Grows linearly with sequence length.
- **Activations:** Intermediate computation results during the forward pass.
- **Framework Overhead:** PyTorch/CUDA runtime, CUDA kernels, memory allocator fragmentation.
- **Input/Output Buffers:** Tokenized input sequences and generated output tensors.
- **OS & Background Processes:** Typically 2-3 GB on most systems.

**3. Would quantization always be the safer choice? Why or why not?**  
Not always. Quantization reduces memory usage but comes with trade-offs:
- **Pros:** Dramatically reduces memory, enables running larger models on consumer hardware, faster inference on some hardware.
- **Cons:** Slight quality degradation (especially aggressive INT4), some tasks requiring high precision (math, code) may suffer, not all models support quantization well, and the quantization process itself can introduce errors.
- For most general-purpose tasks, INT4/INT8 quantization provides an excellent quality-to-efficiency trade-off.

---
## Activity 5 — Run an LLM Locally

**Task:** Run a model locally using Hugging Face Transformers (as our local inference engine). Answer the guided questions.

*Note: We are using HuggingFace Transformers `pipeline` instead of Ollama, which serves the same purpose — running inference locally on our own hardware.*

In [8]:
# Using HuggingFace pipeline as local inference (equivalent to Ollama)
local_generator = pipeline(
    "text-generation",
    model="distilgpt2",
    device=0 if torch.cuda.is_available() else -1,
    framework="pt",
)

prompt = "The future of artificial intelligence is"

result = local_generator(
    prompt,
    max_new_tokens=80,
    do_sample=True,
    temperature=0.7,
    num_return_sequences=1,
    pad_token_id=local_generator.tokenizer.eos_token_id,
)

print("Prompt:", prompt)
print("\nGenerated Response:")
print(result[0]["generated_text"])

# Verify it's running locally
print("\n--- Inference Details ---")
print(f"Model device: {local_generator.device}")
print(f"Running locally: Yes (no API calls made)")
print(f"Internet required: No (model is cached locally)")

Device set to use cuda:0


Prompt: The future of artificial intelligence is

Generated Response:
The future of artificial intelligence is a matter of public awareness, not just in America, but across the globe. However, many experts are skeptical that artificial intelligence is the answer.




















































--- Inference Details ---
Model device: cuda:0
Running locally: Yes (no API calls made)
Internet required: No (model is cached locally)


### Activity 5 — Answers

**1. Does the model need internet after downloading?**  
No. Once the model weights are downloaded and cached locally (in `~/.cache/huggingface/`), the model runs entirely offline. No internet connection is needed for inference.

**2. Where is inference actually happening?**  
Inference is happening on our local machine — specifically on the NVIDIA RTX 4050 GPU. The model weights are loaded into GPU VRAM, and all matrix multiplications and token generation happen on the GPU hardware itself.

**3. Is any data leaving your machine?**  
No. All inputs and outputs stay entirely on the local machine. Unlike cloud API calls, nothing is sent over the network. This makes local inference ideal for sensitive or private data.

**4. What happens if the model is too large for your memory?**  
If a model exceeds available VRAM, several things can happen:
- **OOM Error:** PyTorch throws a `torch.cuda.OutOfMemoryError` and the process crashes.
- **CPU Offloading:** With libraries like `accelerate`, some layers can be offloaded to system RAM (slower but functional).
- **Quantization:** Reducing precision (FP16 → INT8 → INT4) can shrink the model to fit.
- **Disk Offloading:** In extreme cases, parts of the model can be offloaded to disk (very slow).

---
## Activity 6 — Use an LLM Through an API / Inference Provider

**Task:** Call a model through an API provider. Here we demonstrate the concept of API-based inference and compare it with local execution.

*Note: We demonstrate the concept of API-style usage by comparing local pipeline calls with what an API call would look like, and discuss the architectural differences.*

In [9]:
# Demonstrating API-style usage concept
# In a real API scenario, you would use:
# from huggingface_hub import InferenceClient
# client = InferenceClient(token="hf_xxx")
# response = client.text_generation("prompt", model="meta-llama/...")

# For this demo, we use our local pipeline but discuss the API architecture

prompt = "Explain machine learning in one sentence:"

# Local inference (what we're actually doing)
result = local_generator(
    prompt,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.7,
    pad_token_id=local_generator.tokenizer.eos_token_id,
)

print("Local Inference Result:")
print(result[0]["generated_text"])

print("\n" + "=" * 60)
print("API vs Local Inference Comparison:")
print("=" * 60)
print()
print(f"{'Aspect':<22} {'Local (Our Setup)':<28} {'API (Cloud)'}")
print("-" * 80)
print(f"{'Where it runs':<22} {'Our GPU (RTX 4050)':<28} {'Provider data center'}")
print(f"{'Internet needed':<22} {'No (after download)':<28} {'Yes (every request)'}")
print(f"{'Data privacy':<22} {'Full privacy':<28} {'Data sent to provider'}")
print(f"{'Cost':<22} {'Free (own hardware)':<28} {'Pay per token/request'}")
print(f"{'Max model size':<22} {'Limited by VRAM (6GB)':<28} {'No limit (provider scales)'}")
print(f"{'Latency':<22} {'Low (no network)':<28} {'Higher (network round-trip)'}")

Local Inference Result:
Explain machine learning in one sentence: http://www.vimeo.com/578935/

API vs Local Inference Comparison:

Aspect                 Local (Our Setup)            API (Cloud)
--------------------------------------------------------------------------------
Where it runs          Our GPU (RTX 4050)           Provider data center
Internet needed        No (after download)          Yes (every request)
Data privacy           Full privacy                 Data sent to provider
Cost                   Free (own hardware)          Pay per token/request
Max model size         Limited by VRAM (6GB)        No limit (provider scales)
Latency                Low (no network)             Higher (network round-trip)


### Activity 6 — Answers

**1. What is actually happening between your program and the model?**  
In an API call, your program sends an HTTP request (containing the prompt text) over the internet to the provider's server. The server loads the model (or routes to a running instance), runs inference on their hardware (usually powerful GPUs/TPUs), and sends the generated text back as an HTTP response. Your program never touches the model weights directly.

**2. Who is paying for compute here?**  
The API provider pays for the hardware, electricity, and cooling. They pass these costs to you through per-token or per-request pricing. In contrast, with local inference (like our setup), you pay upfront for hardware and electricity, but inference itself is "free" after that.

**3. What are the trade-offs versus running locally?**  
- **API Pros:** Access to very large models (70B+), no hardware requirements, automatic scaling, always up-to-date models.
- **API Cons:** Costs money per request, requires internet, data leaves your machine (privacy concern), latency from network calls, rate limits.
- **Local Pros:** Free after hardware purchase, full privacy, no internet needed, no rate limits, low latency.
- **Local Cons:** Limited by your hardware (can't run 70B on 6GB VRAM), you manage updates, no auto-scaling.

---
## Activity 7 — Compare Local vs Colab vs API

**Task:** Compare the three approaches for running LLMs.

In [10]:
print("Comparison: Local vs Colab vs API")
print("=" * 85)
print()
print(f"{'Criteria':<22} {'Local (Laptop)':<22} {'Google Colab':<22} {'API Provider'}")
print("-" * 85)
print(f"{'Hardware Needed':<22} {'GPU (RTX 4050, 6GB)':<22} {'Free T4 GPU (15GB)':<22} {'None (cloud)'}")
print(f"{'Max Model Size':<22} {'~4B INT4':<22} {'~7B FP16':<22} {'70B+ (no limit)'}")
print(f"{'Cost':<22} {'Free (own HW)':<22} {'Free tier available':<22} {'Pay per token'}")
print(f"{'Privacy':<22} {'Fully local':<22} {'Google cloud':<22} {'Provider cloud'}")
print(f"{'Internet Required':<22} {'No (after download)':<22} {'Yes (always)':<22} {'Yes (always)'}")
print(f"{'Setup Difficulty':<22} {'Medium':<22} {'Easy':<22} {'Easy'}")
print(f"{'Scalability':<22} {'Low (1 GPU)':<22} {'Limited':<22} {'High (auto-scale)'}")
print(f"{'Session Persistence':<22} {'Always on':<22} {'Times out':<22} {'Always on'}")

Comparison: Local vs Colab vs API

Criteria               Local (Laptop)         Google Colab           API Provider
-------------------------------------------------------------------------------------
Hardware Needed        GPU (RTX 4050, 6GB)    Free T4 GPU (15GB)     None (cloud)
Max Model Size         ~4B INT4               ~7B FP16               70B+ (no limit)
Cost                   Free (own HW)          Free tier available    Pay per token
Privacy                Fully local            Google cloud           Provider cloud
Internet Required      No (after download)    Yes (always)           Yes (always)
Setup Difficulty       Medium                 Easy                   Easy
Scalability            Low (1 GPU)            Limited                High (auto-scale)
Session Persistence    Always on              Times out              Always on


### Activity 7 — Answers

**1. If you had an 8 GB RAM laptop with no GPU, which approach would you pick?**  
I would pick **Google Colab** for development/experimentation (free GPU access) and **API** for production use. Running locally on 8 GB RAM with no GPU would only allow very small models (1-3B INT4) with slow CPU inference. Colab gives you a free T4 GPU with 15 GB VRAM, which can handle 7B models comfortably.

**2. Which approach is most private? Which is most scalable?**  
- **Most Private:** Local inference — data never leaves your machine. No third party sees your prompts or outputs.
- **Most Scalable:** API — providers like OpenAI, Google, or HuggingFace can handle millions of requests by scaling their infrastructure automatically. Local setups are limited to one GPU.

**3. When would you choose local over cloud, or the reverse?**  
- **Choose Local when:** Working with sensitive/private data (medical records, legal documents), need offline capability, want zero ongoing costs, or need very low latency for real-time applications.
- **Choose Cloud/API when:** Need access to large models (70B+), need to scale to many users, don't have powerful hardware, or need the latest model versions without manual updates.

---
## Activity 8 — Temperature Experiment

**Task:** Run the same prompt at temperature 0.1, 0.7, and 1.0. Compare the outputs.

In [11]:
prompt = "The best programming language for beginners is"

temperatures = [0.1, 0.7, 1.0]

print(f"Prompt: \"{prompt}\"")
print("=" * 70)

for temp in temperatures:
    print(f"\n--- Temperature: {temp} ---")
    
    # Generate 3 outputs at each temperature to show variability
    for run in range(3):
        result = local_generator(
            prompt,
            max_new_tokens=40,
            do_sample=True,
            temperature=temp,
            top_p=0.95,
            num_return_sequences=1,
            pad_token_id=local_generator.tokenizer.eos_token_id,
        )
        text = result[0]["generated_text"]
        # Only show the generated part (after prompt)
        generated = text[len(prompt):]
        print(f"  Run {run+1}: ...{generated.strip()[:100]}")
    
print()

Prompt: "The best programming language for beginners is"

--- Temperature: 0.1 ---
  Run 1: ...the C++ language. It is a great language for beginners and beginners. It is a great language for beg
  Run 2: ...the C# language. It is a great language for beginners and beginners. It is a great language for begi
  Run 3: ...the C++ language. It is a great language for beginners and beginners. It is a great language for beg

--- Temperature: 0.7 ---
  Run 1: ...Python.
  Run 2: ...the Python 3.6 version. The Python 3.6 version includes several improvements.
  Run 3: ...Python. Python is a popular programming language that has been around for over a decade. It has been

--- Temperature: 1.0 ---
  Run 1: ...Rust and other programming languages that will learn the basics of using it.



The following progra


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  Run 2: ...Ruby, the language that I call Swift, which is inspired by Ruby and is highly recommended for beginn
  Run 3: ...Java, and the first to use Java's C++ language to teach code. And for newcomers to C++, it's best to



### Activity 8 — Answers

**1. Which output felt most predictable? Most creative?**  
- **Most Predictable:** Temperature 0.1 — The outputs are nearly identical across all 3 runs. The model picks the highest-probability token almost every time, producing deterministic, repetitive results.
- **Most Creative:** Temperature 1.0 — Each run produces noticeably different text. The model explores lower-probability tokens, leading to more diverse and sometimes surprising word choices.

**2. Does temperature change what the model knows?**  
No! Temperature does NOT change the model's knowledge, weights, or training. It only changes the *sampling strategy* — how the model selects the next token from its probability distribution:
- Low temperature → sharpens the distribution → model always picks the "most likely" token.
- High temperature → flattens the distribution → model is more willing to pick less likely tokens.

**3. When would you want low temperature? High temperature?**  
- **Low temperature (0.1-0.3):** Factual Q&A, code generation, data extraction, medical/legal text — where accuracy and consistency matter.
- **Medium temperature (0.5-0.7):** General conversation, summarization, explanations — balanced between coherence and variety.
- **High temperature (0.8-1.0+):** Creative writing, brainstorming, poetry, story generation — where novelty and diversity are desired.

---
## Activity 9 — Build a Mini Chatbot

**Task:** Build a chatbot loop using the local model, keeping conversation history.

In [12]:
class MiniChatbot:
    """A simple chatbot that maintains conversation history."""
    
    def __init__(self, generator, max_history_tokens=500):
        self.generator = generator
        self.history = []
        self.max_history_tokens = max_history_tokens
    
    def chat(self, user_message):
        """Send a message and get a response."""
        # Add user message to history
        self.history.append(f"User: {user_message}")
        
        # Build the full conversation context
        conversation = "\n".join(self.history) + "\nAssistant:"
        
        # Generate response
        result = self.generator(
            conversation,
            max_new_tokens=60,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=self.generator.tokenizer.eos_token_id,
        )
        
        # Extract only the new generated text
        full_output = result[0]["generated_text"]
        response = full_output[len(conversation):].strip()
        
        # Clean up — take only the first meaningful sentence/line
        response = response.split("\n")[0].strip()
        if not response:
            response = "I'm not sure how to respond to that."
        
        # Add assistant response to history
        self.history.append(f"Assistant: {response}")
        
        return response
    
    def clear_history(self):
        """Reset conversation history."""
        self.history = []
        print("Conversation history cleared.")

# Create the chatbot
chatbot = MiniChatbot(local_generator)

# Simulate a conversation
test_messages = [
    "Hello! What is machine learning?",
    "Can you give me an example?",
    "What about deep learning?",
]

print("=" * 60)
print("Mini Chatbot Conversation")
print("=" * 60)

for msg in test_messages:
    print(f"\nUser: {msg}")
    response = chatbot.chat(msg)
    print(f"Bot:  {response}")

# Show the conversation history
print("\n" + "=" * 60)
print("Full Conversation History:")
print("=" * 60)
for entry in chatbot.history:
    print(entry)

Mini Chatbot Conversation

User: Hello! What is machine learning?
Bot:  Machine learning is an art form for people who think about their everyday tasks. This is not a new topic. Machine learning is a form of thinking about the human condition. It is not just a way to learn how to solve problems. Machine learning is also an art form of thinking about the human condition

User: Can you give me an example?
Bot:  Well, I can give you a quick example of how to solve problems, and I will give you an example of how to solve problems.

User: What about deep learning?
Bot:  It is a very important thing to know that deep learning is the most important thing to know. Deep learning is a new field that is growing faster than previous techniques. In the past, deep learning has been used to solve problems. Nowadays, we are using deep learning to solve a problem.

Full Conversation History:
User: Hello! What is machine learning?
Assistant: Machine learning is an art form for people who think about the

In [13]:
# Demonstrate what happens WITHOUT conversation history
print("=" * 60)
print("WITHOUT History (each message is independent):")
print("=" * 60)

messages_without_history = [
    "Hello! What is machine learning?",
    "Can you give me an example?",  # Without history, "example of what?" is ambiguous
    "What about deep learning?",
]

for msg in messages_without_history:
    # Each message is treated independently — no context
    result = local_generator(
        f"User: {msg}\nAssistant:",
        max_new_tokens=60,
        do_sample=True,
        temperature=0.7,
        pad_token_id=local_generator.tokenizer.eos_token_id,
    )
    response = result[0]["generated_text"].split("Assistant:")[-1].strip().split("\n")[0]
    print(f"\nUser: {msg}")
    print(f"Bot:  {response}")

WITHOUT History (each message is independent):

User: Hello! What is machine learning?
Bot:  Computer science (computer science)

User: Can you give me an example?
Bot:  Not really. But that's not the best part. I really just need to be able to talk to my client in real time.

User: What about deep learning?
Bot:  I really just don't think about deep learning.


### Activity 9 — Answers

**1. Why does the chatbot need to remember previous messages?**  
Without history, the chatbot treats every message as an isolated question. When the user says "Can you give me an example?", the chatbot with history knows we were discussing machine learning and provides a relevant ML example. Without history, it has no idea what "an example" refers to — the question is completely ambiguous.

**2. What happens if you drop the conversation history?**  
Each response becomes independent and context-free. Follow-up questions like "tell me more" or "what about that?" become meaningless because the model has no memory of what "that" refers to. The conversation feels disjointed and frustrating.

**3. What's the difference between this and a single one-off prompt?**  
A single prompt gets one response with no follow-up capability. A chatbot maintains a running conversation where each new message includes all previous context, allowing for:
- Follow-up questions and clarifications
- Building on previous answers
- A natural, flowing dialogue experience
- The ability to refine or redirect the conversation

---
## Activity 10 — Customize the Chatbot

**Task:** Give the chatbot a role, such as an AI/ML tutor.

In [14]:
class CustomChatbot:
    """A chatbot with a customizable system role."""
    
    def __init__(self, generator, system_prompt):
        self.generator = generator
        self.system_prompt = system_prompt
        self.history = []
    
    def chat(self, user_message):
        self.history.append(f"Student: {user_message}")
        
        # Build context with system prompt + history
        context = f"{self.system_prompt}\n\n"
        context += "\n".join(self.history)
        context += "\nTutor:"
        
        result = self.generator(
            context,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.6,
            top_p=0.9,
            pad_token_id=self.generator.tokenizer.eos_token_id,
        )
        
        full_output = result[0]["generated_text"]
        response = full_output[len(context):].strip().split("\n")[0].strip()
        
        if not response:
            response = "That's a great question! Let me think about that."
        
        self.history.append(f"Tutor: {response}")
        return response

# ── AI/ML Tutor Role ──
tutor_prompt = """You are an AI/ML Tutor. You explain machine learning concepts 
in simple terms with real-world examples. You are patient, encouraging, 
and always relate concepts to everyday life. Keep answers concise."""

tutor_bot = CustomChatbot(local_generator, tutor_prompt)

print("=" * 60)
print("AI/ML Tutor Chatbot")
print("=" * 60)

tutor_questions = [
    "What is overfitting?",
    "How do I prevent it?",
    "What is the difference between supervised and unsupervised learning?",
]

for q in tutor_questions:
    print(f"\nStudent: {q}")
    response = tutor_bot.chat(q)
    print(f"Tutor:   {response}")

AI/ML Tutor Chatbot

Student: What is overfitting?
Tutor:   What is overfitting?

Student: How do I prevent it?
Tutor:   What is overfitting?

Student: What is the difference between supervised and unsupervised learning?
Tutor:   What is the difference between supervised and unsupervised learning?


In [ ]:
# ── Coding Assistant Role ──
coder_prompt = """You are a Python Coding Assistant. You help students write 
clean Python code. You explain code step by step, suggest best practices, 
and fix bugs. Be concise and use code examples."""

coder_bot = CustomChatbot(local_generator, coder_prompt)

print("=" * 60)
print("Python Coding Assistant Chatbot")
print("=" * 60)

coder_questions = [
    "How do I read a CSV file in Python?",
    "What is a list comprehension?",
    "How do I handle errors in Python?",
]

for q in coder_questions:
    print(f"\nStudent: {q}")
    response = coder_bot.chat(q)
    print(f"Assistant: {response}")

Python Coding Assistant Chatbot

Student: How do I read a CSV file in Python?
Assistant: You can read the CSV file in Python using the Python Coding Assistant. You can also read the CSV file in Python using the Python Coding Assistant.

Student: What is a list comprehension?
Assistant: You can use a list comprehension. You can use a list comprehension. You can use a list comprehension. You can use a list comprehension. You can use a list comprehension. You can use a list comprehension. You can use a list comprehension. You can use a list comprehension. You can use a list comprehension. You can use a list comprehension. You can use a list comprehension. You can use

Student: How do I handle errors in Python?
Assistant: You can use a list comprehension. You can use a list comprehension. You can use a list comprehension. You can use a list comprehension. You can use a list comprehension. You can use a list comprehension. You can use a list comprehension. You can use a list comprehension. 

: 

### Activity 10 — Answers

**1. How did the responses change compared to Activity 9?**  
The responses became much more focused and topic-appropriate. The AI/ML Tutor explains concepts in simple language with encouragement, while the Coding Assistant provides practical code examples and debugging tips. Without a role, the generic chatbot gives unfocused, general responses.

**2. Could the same model behave like a completely different assistant just from the instructions you gave it?**  
Yes! The same distilgpt2 model behaves completely differently depending on the system prompt:
- As an AI/ML Tutor: Patient, encouraging, uses analogies
- As a Coding Assistant: Technical, concise, provides code snippets
This demonstrates the power of **prompt engineering** — the model's behavior is shaped by the context and instructions provided, not by retraining.

**3. What role would you build next, and why?**  
I would build a **Data Analysis Assistant** that helps users explore datasets, suggest visualizations, write pandas code, and interpret statistical results. This would be practical for students and data scientists who need quick guidance while working with data, combining coding help with domain knowledge.

---
## Summary

All 10 activities have been completed using **HuggingFace Transformers** running locally on an **NVIDIA RTX 4050 GPU** with **6 GB VRAM**. Key takeaways:

1. **Model Exploration:** LLMs range from 82M to 70B+ parameters, with trade-offs between capability and resource requirements.
2. **Tokenization:** Text is split into sub-word tokens using BPE; not every word is a single token.
3. **GPU Inference:** Even a small GPU can run compact models efficiently.
4. **Memory Estimation:** Understanding FP32/FP16/INT8/INT4 precision is crucial for hardware planning.
5. **Local vs Cloud:** Privacy vs scalability is the fundamental trade-off.
6. **Temperature:** Controls randomness in generation without changing model knowledge.
7. **Chatbot History:** Context/memory is essential for coherent multi-turn conversations.
8. **Role Customization:** System prompts can dramatically change model behavior without retraining.